# Day 2: Industrial-Grade CausalSelfAttention (nn.Module)
### Assembling All Day 1 Fundamentals into a Production-Grade PyTorch Class

In Day 1, we executed each tensor step with raw functions. Today, we assemble them into an object-oriented PyTorch `nn.Module`.

### 3 Key Engineering Patterns You Will Master:
1. **Fused QKV Projection (`nn.Linear(C, 3 * C)`)**: Computing $Q, K, V$ in one single kernel launch.
2. **Persistent Non-Trainable Buffers (`register_buffer`)**: Storing the causal mask cleanly.
3. **Output Projection (`c_proj`)**: Mixing the heads back into the residual stream.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")
if device.type == "cuda":
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")

Running on device: cuda
Active GPU: NVIDIA GeForce RTX 5070


In [2]:
nn.Linear(16, 3*16)

Linear(in_features=16, out_features=48, bias=True)

## 1. Defining the `CausalSelfAttention` Class

In [3]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd: int, n_head: int, block_size: int, dropout: float = 0.1):
        super().__init__()
        assert n_embd % n_head == 0, f"n_embd ({n_embd}) must be divisible by n_head ({n_head})"

        self.n_embd = n_embd
        self.n_head = n_head
        self.head_dim = n_embd // n_head

        # 1. FUSED QKV PROJECTION: C -> 3 * C
        self.c_attn = nn.Linear(n_embd, 3 * n_embd)

        # 2. OUTPUT PROJECTION: C -> C
        self.c_proj = nn.Linear(n_embd, n_embd)

        # 3. REGULARIZATION
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        # 4. CAUSAL MASK BUFFER
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("bias", mask.view(1, 1, block_size, block_size))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape

        # Step 1: Fused QKV and head splitting
        # (B, T, C) -> (B, T, 3 * C) -> 3 chunks of (B, T, C)
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # Transpose to (B, nh, T, hs)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Step 2: Attention scores (B, nh, T, T)
        scale = 1.0 / math.sqrt(self.head_dim)
        scores = (q @ k.transpose(-2, -1)) * scale
        scores = scores.masked_fill(self.bias[:, :, :T, :T] == 0, float("-inf"))

        # Softmax across columns (dim=-1) -> each row sums to 1.0
        weights = F.softmax(scores, dim=-1)
        weights = self.attn_dropout(weights)

        # Step 3: Weighted sum of Values (B, nh, T, hs)
        y = weights @ v

        # Step 4: Reassemble heads (B, T, C) and output projection
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

## 2. Instantiating & Inspecting Parameters

In [4]:
B, T, C, nh = 2, 6, 64, 4
model = CausalSelfAttention(n_embd=C, n_head=nh, block_size=32).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

for name, param in model.named_parameters():
    print(f"  {name:15s} -> Shape: {str(param.shape):20s} ({param.numel():,} params)")

print(f"\nRegistered buffer (bias mask) shape: {model.bias.shape}")

Total trainable parameters: 16,640
  c_attn.weight   -> Shape: torch.Size([192, 64]) (12,288 params)
  c_attn.bias     -> Shape: torch.Size([192])    (192 params)
  c_proj.weight   -> Shape: torch.Size([64, 64]) (4,096 params)
  c_proj.bias     -> Shape: torch.Size([64])     (64 params)

Registered buffer (bias mask) shape: torch.Size([1, 1, 32, 32])


## 3. Forward Pass & Autograd Verification

In [5]:
# Create dummy input batch (2 sentences, 6 tokens, 64 features)
x = torch.randn(B, T, C, device=device, requires_grad=True)

# Forward pass
out = model(x)
print(f"Input shape : {x.shape}")
print(f"Output shape: {out.shape} (Matches exactly!)")

# Backward pass
loss = out.sum()
loss.backward()

print(f"\nGradient checks:")
print(f"- x.grad magnitude norm: {x.grad.norm():.4f}")
print(f"- c_attn.weight.grad norm: {model.c_attn.weight.grad.norm():.4f}")
print(f"- c_proj.weight.grad norm: {model.c_proj.weight.grad.norm():.4f}")
print("\nAll gradients computed cleanly through CausalSelfAttention!")

Input shape : torch.Size([2, 6, 64])
Output shape: torch.Size([2, 6, 64]) (Matches exactly!)

Gradient checks:
- x.grad magnitude norm: 10.4528
- c_attn.weight.grad norm: 142.1819
- c_proj.weight.grad norm: 215.5954

All gradients computed cleanly through CausalSelfAttention!
